# PPE YOLOv8 training (Colab / Kaggle)

Use **this notebook** for E0–E4. Local 8GB GPUs are only for baseline val, export, and `--batch 8` smokes.

Product bars: **vest / no_vest 95%+**, helmets next, goggles ~70% OK. **Boots are not in this cycle.**

1. Runtime → GPU (Colab) or GPU accelerator (Kaggle).
2. Add secret `ROBOFLOW_API_KEY` (Colab userdata / Kaggle Add-ons → Secrets).
3. Run all cells. Default experiment: `e0_n` on the 12k subset after Combined download + remap.

Prefer **one** of Colab or Kaggle, not both.

In [ ]:
import os
from pathlib import Path

# Colab secret, then Kaggle, then env (local fallback).
try:
    from google.colab import userdata
    os.environ.setdefault("ROBOFLOW_API_KEY", userdata.get("ROBOFLOW_API_KEY"))
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ.setdefault("ROBOFLOW_API_KEY", UserSecretsClient().get_secret("ROBOFLOW_API_KEY"))
    except Exception:
        pass

assert os.environ.get("ROBOFLOW_API_KEY"), "Set ROBOFLOW_API_KEY as a Colab/Kaggle secret"
print("key_set", "colab" if IN_COLAB else "kaggle_or_local")

REPO = Path("/content/ppe") if IN_COLAB else Path("/kaggle/working/ppe")
if not (REPO / "scripts" / "train.py").exists():
    REPO.mkdir(parents=True, exist_ok=True)
    !git clone --depth 1 https://github.com/A-Kuo/Worker-Safety-PPE-Detection-Model.git {REPO}
os.chdir(REPO)
print("cwd", Path.cwd())

In [ ]:
%pip install -q ultralytics roboflow pyyaml opencv-python-headless
%pip install -q -e .
import torch
print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

In [ ]:
# Combined (~2.4GB zip) then Hard Hat Universe. Construction is optional for mapped eval.
!python scripts/download_datasets.py --execute --only combined hardhat
!python scripts/remap_labels.py --source data/raw/combined --out data/processed/combined --mapping combined
!python scripts/remap_labels.py --source data/raw/hardhat --out data/processed/hardhat --mapping hhu
!python scripts/make_subset.py --source data/processed/combined --out data/raw/combined_12k --n 12000 --seed 42
!python scripts/analyze_distribution.py

In [ ]:
# E0 on 12k. Swap --exp: e1_s | e2_focal | e3_augs | e4_full44k
# P100/T4: batch 16. If OOM, add --batch 8.
!python scripts/train.py --exp e0_n --device 0 --batch 16

After E0, copy `runs/train/e0_n/weights/best.pt` off the VM (Drive / Kaggle output). Then run `eval.py` / `calibrate.py` and lead the report with **vest / no_vest**.